# Interactive Linkage Mapping Tutorial
## Understanding 2-Point and 3-Point Crosses

**Learning Objectives:**
- Understand genetic linkage and recombination
- Calculate map distances using recombination frequencies
- Determine gene order using 3-point crosses
- Visualize chromosomal arrangements and crossover events

---

In [ ]:
# Install required packages if needed
!pip install ipywidgets matplotlib numpy pandas seaborn -q

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, interactive_output, VBox, HBox, Label
import seaborn as sns
from IPython.display import display, Markdown, HTML

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Part 1: Understanding Genetic Linkage

**Key Concepts:**
- Genes on the same chromosome are **linked**
- **Recombination frequency (RF)** = (Number of recombinants / Total offspring) × 100
- **1 map unit (m.u.) = 1 centiMorgan (cM) = 1% recombination**
- Closer genes have lower recombination frequencies

---
## Part 2: Two-Point Cross Analysis

In a two-point cross, we study two genes to determine:
1. Whether they are linked or independently assorting
2. The map distance between them

In [ ]:
def calculate_two_point_cross(parental_count, recombinant_count):
    """
    Calculate recombination frequency and map distance for a two-point cross
    """
    total = parental_count + recombinant_count
    rf = (recombinant_count / total) * 100
    map_distance = rf  # In centiMorgans
    
    return rf, map_distance, total

def visualize_two_point_cross(parental_count, recombinant_count):
    """
    Visualize a two-point cross with interactive parameters
    """
    rf, map_dist, total = calculate_two_point_cross(parental_count, recombinant_count)
    
    # Create figure with subplots
    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.4, wspace=0.3)
    
    # 1. Bar chart of offspring types
    ax1 = fig.add_subplot(gs[0, :])
    categories = ['Parental Types', 'Recombinant Types']
    counts = [parental_count, recombinant_count]
    colors = ['#2E86AB', '#A23B72']
    bars = ax1.bar(categories, counts, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    ax1.set_ylabel('Number of Offspring', fontsize=12, fontweight='bold')
    ax1.set_title('Two-Point Cross: Offspring Distribution', fontsize=14, fontweight='bold')
    
    # Add count labels on bars
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{count}\n({count/total*100:.1f}%)',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # 2. Chromosome diagram
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 5)
    ax2.axis('off')
    ax2.set_title('Chromosome Map', fontsize=12, fontweight='bold')
    
    # Draw chromosome
    chromosome_y = 2.5
    ax2.plot([1, 9], [chromosome_y, chromosome_y], 'k-', linewidth=8)
    
    # Gene A position
    ax2.plot(2, chromosome_y, 'ro', markersize=20, label='Gene A')
    ax2.text(2, chromosome_y + 0.7, 'Gene A', ha='center', fontsize=11, fontweight='bold')
    
    # Gene B position (distance proportional to map distance)
    gene_b_pos = 2 + (map_dist / 50) * 6  # Scale to fit nicely
    ax2.plot(gene_b_pos, chromosome_y, 'bo', markersize=20, label='Gene B')
    ax2.text(gene_b_pos, chromosome_y + 0.7, 'Gene B', ha='center', fontsize=11, fontweight='bold')
    
    # Distance annotation
    ax2.annotate('', xy=(gene_b_pos, chromosome_y - 0.5), xytext=(2, chromosome_y - 0.5),
                arrowprops=dict(arrowstyle='<->', color='green', lw=2))
    ax2.text((2 + gene_b_pos)/2, chromosome_y - 1, f'{map_dist:.2f} cM',
            ha='center', fontsize=11, fontweight='bold', color='green')
    
    # 3. Crossover diagram
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.set_xlim(0, 10)
    ax3.set_ylim(0, 6)
    ax3.axis('off')
    ax3.set_title('Crossover Events', fontsize=12, fontweight='bold')
    
    # Draw two homologous chromosomes
    y1, y2 = 4, 2
    ax3.plot([1, 9], [y1, y1], 'b-', linewidth=6, label='Chromosome 1', alpha=0.7)
    ax3.plot([1, 9], [y2, y2], 'r-', linewidth=6, label='Chromosome 2', alpha=0.7)
    
    # Show crossover
    crossover_x = 5
    ax3.plot([crossover_x, crossover_x], [y2, y1], 'g--', linewidth=3, label='Crossover')
    ax3.text(crossover_x + 0.5, 3, 'Crossover\nRegion', fontsize=10, fontweight='bold', color='green')
    
    # Gene markers
    ax3.plot([2, 2], [y1, y2], 'ko', markersize=10)
    ax3.plot([7, 7], [y1, y2], 'ko', markersize=10)
    ax3.text(2, y1 + 0.4, 'A', ha='center', fontsize=10, fontweight='bold')
    ax3.text(7, y1 + 0.4, 'B', ha='center', fontsize=10, fontweight='bold')
    
    ax3.legend(loc='upper right')
    
    # 4. Results summary
    ax4 = fig.add_subplot(gs[2, :])
    ax4.axis('off')
    
    # Determine linkage
    if rf < 50:
        linkage_status = "LINKED"
        linkage_color = "green"
        interpretation = "Genes are on the same chromosome and show linkage."
    else:
        linkage_status = "INDEPENDENT"
        linkage_color = "orange"
        interpretation = "Genes assort independently (either far apart or on different chromosomes)."
    
    summary_text = f"""
    ANALYSIS RESULTS:
    
    Total Offspring: {total}
    Parental Types: {parental_count} ({parental_count/total*100:.1f}%)
    Recombinant Types: {recombinant_count} ({recombinant_count/total*100:.1f}%)
    
    Recombination Frequency: {rf:.2f}%
    Map Distance: {map_dist:.2f} centiMorgans (cM)
    
    Linkage Status: {linkage_status}
    Interpretation: {interpretation}
    """
    
    ax4.text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    plt.tight_layout()
    plt.show()
    
    # Print additional educational information
    print("\n" + "="*60)
    print("INTERPRETATION GUIDE:")
    print("="*60)
    if rf < 10:
        print("✓ Genes are VERY CLOSELY linked (< 10 cM apart)")
        print("  Crossovers are rare between these genes.")
    elif rf < 30:
        print("✓ Genes are MODERATELY linked (10-30 cM apart)")
        print("  Some crossovers occur, but parental types still dominate.")
    elif rf < 50:
        print("✓ Genes are LOOSELY linked (30-50 cM apart)")
        print("  Significant recombination occurs.")
    else:
        print("✓ Genes show INDEPENDENT assortment (≥ 50 cM or different chromosomes)")
        print("  Recombinants and parentals are equally frequent.")

### Interactive Two-Point Cross Simulator

**Instructions:**
- Adjust the sliders to change the number of parental and recombinant offspring
- Observe how the recombination frequency and map distance change
- Try to understand the relationship between gene distance and recombination

In [ ]:
# Interactive widget for two-point cross
interact(visualize_two_point_cross,
         parental_count=IntSlider(min=100, max=900, step=50, value=700, description='Parental:'),
         recombinant_count=IntSlider(min=10, max=500, step=10, value=100, description='Recombinant:'));

---
## Part 3: Three-Point Cross Analysis

A three-point cross involves three linked genes. It allows us to:
1. Determine the **gene order** on the chromosome
2. Calculate **map distances** between genes
3. Detect **double crossovers**
4. Calculate **coefficient of coincidence (COC)** and **interference**

In [ ]:
def analyze_three_point_cross(nco, sco_ab, sco_bc, dco, total_offspring):
    """
    Analyze a three-point cross
    
    Parameters:
    - nco: No crossover (parental types)
    - sco_ab: Single crossover between genes A and B
    - sco_bc: Single crossover between genes B and C
    - dco: Double crossover
    - total_offspring: Total number of offspring
    """
    
    # Calculate recombination frequencies
    rf_ab = ((sco_ab + dco) / total_offspring) * 100
    rf_bc = ((sco_bc + dco) / total_offspring) * 100
    rf_ac = ((sco_ab + sco_bc + 2*dco) / total_offspring) * 100
    
    # Map distances
    map_ab = rf_ab
    map_bc = rf_bc
    map_ac = map_ab + map_bc
    
    # Coefficient of coincidence and interference
    expected_dco = (rf_ab / 100) * (rf_bc / 100) * total_offspring
    coc = dco / expected_dco if expected_dco > 0 else 0
    interference = 1 - coc
    
    return {
        'rf_ab': rf_ab,
        'rf_bc': rf_bc,
        'rf_ac': rf_ac,
        'map_ab': map_ab,
        'map_bc': map_bc,
        'map_ac': map_ac,
        'expected_dco': expected_dco,
        'observed_dco': dco,
        'coc': coc,
        'interference': interference
    }

def visualize_three_point_cross(nco, sco_ab, sco_bc, dco):
    """
    Visualize three-point cross analysis
    """
    total = nco + sco_ab + sco_bc + dco
    results = analyze_three_point_cross(nco, sco_ab, sco_bc, dco, total)
    
    # Create figure
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(4, 2, hspace=0.4, wspace=0.3)
    
    # 1. Offspring distribution
    ax1 = fig.add_subplot(gs[0, :])
    categories = ['No Crossover\n(Parental)', 'SCO Region A-B', 'SCO Region B-C', 'Double Crossover']
    counts = [nco, sco_ab, sco_bc, dco]
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
    
    bars = ax1.bar(categories, counts, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    ax1.set_ylabel('Number of Offspring', fontsize=12, fontweight='bold')
    ax1.set_title('Three-Point Cross: Offspring Distribution by Crossover Type', fontsize=14, fontweight='bold')
    
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{count}\n({count/total*100:.1f}%)',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 2. Chromosome map with three genes
    ax2 = fig.add_subplot(gs[1, :])
    ax2.set_xlim(0, 100)
    ax2.set_ylim(0, 10)
    ax2.axis('off')
    ax2.set_title('Three-Gene Chromosome Map (Gene Order: A - B - C)', fontsize=13, fontweight='bold')
    
    # Draw chromosome
    chrom_y = 5
    ax2.plot([10, 90], [chrom_y, chrom_y], 'k-', linewidth=10)
    
    # Gene positions (scaled)
    total_map = results['map_ac']
    scale_factor = 60 / max(total_map, 1)  # Scale to fit in 60 units
    
    gene_a_pos = 15
    gene_b_pos = gene_a_pos + results['map_ab'] * scale_factor
    gene_c_pos = gene_b_pos + results['map_bc'] * scale_factor
    
    # Plot genes
    ax2.plot(gene_a_pos, chrom_y, 'ro', markersize=25, label='Gene A')
    ax2.plot(gene_b_pos, chrom_y, 'go', markersize=25, label='Gene B')
    ax2.plot(gene_c_pos, chrom_y, 'bo', markersize=25, label='Gene C')
    
    ax2.text(gene_a_pos, chrom_y + 1.5, 'A', ha='center', fontsize=14, fontweight='bold')
    ax2.text(gene_b_pos, chrom_y + 1.5, 'B', ha='center', fontsize=14, fontweight='bold')
    ax2.text(gene_c_pos, chrom_y + 1.5, 'C', ha='center', fontsize=14, fontweight='bold')
    
    # Distance annotations
    # A to B
    ax2.annotate('', xy=(gene_b_pos, chrom_y - 1.5), xytext=(gene_a_pos, chrom_y - 1.5),
                arrowprops=dict(arrowstyle='<->', color='purple', lw=2.5))
    ax2.text((gene_a_pos + gene_b_pos)/2, chrom_y - 2.3, 
            f'{results["map_ab"]:.2f} cM',
            ha='center', fontsize=11, fontweight='bold', color='purple')
    
    # B to C
    ax2.annotate('', xy=(gene_c_pos, chrom_y - 1.5), xytext=(gene_b_pos, chrom_y - 1.5),
                arrowprops=dict(arrowstyle='<->', color='purple', lw=2.5))
    ax2.text((gene_b_pos + gene_c_pos)/2, chrom_y - 2.3,
            f'{results["map_bc"]:.2f} cM',
            ha='center', fontsize=11, fontweight='bold', color='purple')
    
    # A to C (total)
    ax2.annotate('', xy=(gene_c_pos, chrom_y + 3), xytext=(gene_a_pos, chrom_y + 3),
                arrowprops=dict(arrowstyle='<->', color='orange', lw=2.5))
    ax2.text((gene_a_pos + gene_c_pos)/2, chrom_y + 3.8,
            f'Total: {results["map_ac"]:.2f} cM',
            ha='center', fontsize=11, fontweight='bold', color='orange')
    
    # 3. Crossover type visualization
    ax3 = fig.add_subplot(gs[2, 0])
    ax3.set_xlim(0, 10)
    ax3.set_ylim(0, 12)
    ax3.axis('off')
    ax3.set_title('Crossover Scenarios', fontsize=12, fontweight='bold')
    
    y_positions = [10, 7, 4, 1]
    labels = ['No Crossover', 'SCO (A-B)', 'SCO (B-C)', 'Double CO']
    
    for i, (y_pos, label) in enumerate(zip(y_positions, labels)):
        # Draw chromosome pair
        ax3.plot([1, 9], [y_pos, y_pos], 'k-', linewidth=4, alpha=0.6)
        ax3.plot([1, 9], [y_pos - 0.4, y_pos - 0.4], 'gray', linewidth=4, alpha=0.6)
        
        # Mark genes
        ax3.plot([2, 5, 8], [y_pos, y_pos, y_pos], 'ro', markersize=8)
        
        # Draw crossovers
        if i == 1:  # SCO A-B
            ax3.plot([3.5, 3.5], [y_pos - 0.4, y_pos], 'g-', linewidth=3)
            ax3.text(3.5, y_pos + 0.5, 'X', ha='center', fontsize=12, color='green', fontweight='bold')
        elif i == 2:  # SCO B-C
            ax3.plot([6.5, 6.5], [y_pos - 0.4, y_pos], 'g-', linewidth=3)
            ax3.text(6.5, y_pos + 0.5, 'X', ha='center', fontsize=12, color='green', fontweight='bold')
        elif i == 3:  # Double CO
            ax3.plot([3.5, 3.5], [y_pos - 0.4, y_pos], 'g-', linewidth=3)
            ax3.plot([6.5, 6.5], [y_pos - 0.4, y_pos], 'g-', linewidth=3)
            ax3.text(3.5, y_pos + 0.5, 'X', ha='center', fontsize=12, color='green', fontweight='bold')
            ax3.text(6.5, y_pos + 0.5, 'X', ha='center', fontsize=12, color='green', fontweight='bold')
        
        ax3.text(0.3, y_pos - 0.2, label, ha='right', fontsize=9, fontweight='bold')
    
    # 4. Interference calculation
    ax4 = fig.add_subplot(gs[2, 1])
    ax4.axis('off')
    ax4.set_title('Interference Analysis', fontsize=12, fontweight='bold')
    
    interference_text = f"""
    Expected Double Crossovers:
    = (RF_AB × RF_BC × Total offspring)
    = ({results['rf_ab']:.2f}% × {results['rf_bc']:.2f}% × {total})
    = {results['expected_dco']:.2f}
    
    Observed Double Crossovers: {results['observed_dco']}
    
    Coefficient of Coincidence (COC):
    = Observed DCO / Expected DCO
    = {results['observed_dco']:.0f} / {results['expected_dco']:.2f}
    = {results['coc']:.4f}
    
    Interference:
    = 1 - COC
    = 1 - {results['coc']:.4f}
    = {results['interference']:.4f}
    """
    
    ax4.text(0.1, 0.5, interference_text, fontsize=10, family='monospace',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
    
    # Add interpretation
    if results['interference'] > 0.5:
        interp = "HIGH interference\n(First CO inhibits second)"
        color = 'red'
    elif results['interference'] > 0.2:
        interp = "MODERATE interference"
        color = 'orange'
    else:
        interp = "LOW interference\n(COs independent)"
        color = 'green'
    
    ax4.text(0.5, 0.05, f"Interpretation: {interp}",
            ha='center', fontsize=10, fontweight='bold', color=color,
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))
    
    # 5. Summary table
    ax5 = fig.add_subplot(gs[3, :])
    ax5.axis('off')
    
    summary_data = [
        ['Parameter', 'Value', 'Interpretation'],
        ['Total Offspring', f'{total}', ''],
        ['RF (A-B)', f'{results["rf_ab"]:.2f}%', f'{results["map_ab"]:.2f} cM'],
        ['RF (B-C)', f'{results["rf_bc"]:.2f}%', f'{results["map_bc"]:.2f} cM'],
        ['RF (A-C)', f'{results["rf_ac"]:.2f}%', f'{results["map_ac"]:.2f} cM'],
        ['Gene Order', 'A — B — C', 'B is in the middle'],
        ['Interference', f'{results["interference"]:.4f}', 'See interpretation above']
    ]
    
    table = ax5.table(cellText=summary_data, cellLoc='left', loc='center',
                     colWidths=[0.3, 0.3, 0.4])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    
    # Style header row
    for i in range(3):
        table[(0, i)].set_facecolor('#4CAF50')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Alternate row colors
    for i in range(1, len(summary_data)):
        for j in range(3):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#E8F5E9')
    
    plt.tight_layout()
    plt.show()
    
    # Print educational notes
    print("\n" + "="*70)
    print("KEY CONCEPTS:")
    print("="*70)
    print("1. Gene Order: The gene with the smallest class of recombinants")
    print("   when compared to the other two is in the MIDDLE.")
    print("\n2. Interference: One crossover can prevent another nearby crossover.")
    print("   High interference means fewer double crossovers than expected.")
    print("\n3. Map Distance: Sum of individual distances (A-B + B-C) gives")
    print("   the total map distance from A to C.")

### Interactive Three-Point Cross Simulator

**Instructions:**
- Adjust offspring counts for each crossover class
- Watch how gene distances and interference change
- Try setting high/low double crossover values to see interference effects

In [ ]:
# Interactive widget for three-point cross
interact(visualize_three_point_cross,
         nco=IntSlider(min=100, max=800, step=50, value=500, 
                       description='No Crossover:', style={'description_width': 'initial'}),
         sco_ab=IntSlider(min=20, max=300, step=10, value=100, 
                          description='SCO (A-B):', style={'description_width': 'initial'}),
         sco_bc=IntSlider(min=20, max=300, step=10, value=150, 
                          description='SCO (B-C):', style={'description_width': 'initial'}),
         dco=IntSlider(min=0, max=100, step=5, value=20, 
                       description='Double CO:', style={'description_width': 'initial'}));

---
## Part 4: Practice Problems

Use the simulators above to solve these practice problems:

### Problem 1: Two-Point Cross

You performed a testcross with two genes and obtained the following results:
- Parental types: 750 offspring
- Recombinant types: 250 offspring

**Questions:**
1. What is the recombination frequency?
2. What is the map distance between the genes?
3. Are these genes linked or independently assorting?

*Use the two-point cross simulator above with these values to find the answers.*

### Problem 2: Three-Point Cross

In a three-point cross, you obtained:
- No crossover (parental): 400 offspring
- Single crossover region A-B: 120 offspring
- Single crossover region B-C: 180 offspring
- Double crossover: 30 offspring

**Questions:**
1. What is the gene order?
2. What is the map distance between each pair of genes?
3. Calculate the coefficient of coincidence and interference.
4. Interpret the interference value.

*Use the three-point cross simulator above with these values to find the answers.*

---
## Part 5: Real-World Example - Drosophila

Let's look at a classic example using *Drosophila melanogaster* (fruit fly) genes:

In [ ]:
# Example: Drosophila three-point cross
# Genes: Body color (b), Eye color (pr), Wing length (vg)

print("Classical Drosophila Three-Point Cross")
print("="*50)
print("Genes studied:")
print("  b  = body color (black vs. gray)")
print("  pr = eye color (purple vs. red)")
print("  vg = wing length (vestigial vs. normal)")
print("\nAll three genes are on Chromosome 2")
print("="*50)

# Typical results from a Drosophila cross
visualize_three_point_cross(
    nco=580,      # Parental types (most common)
    sco_ab=90,    # SCO between b and pr
    sco_bc=120,   # SCO between pr and vg
    dco=15        # Double crossovers (rarest)
)

---
## Summary and Learning Outcomes

**What you learned:**

1. **Linkage Mapping Basics**
   - Linked genes don't assort independently
   - Recombination frequency measures genetic distance
   - 1% recombination = 1 map unit = 1 centiMorgan

2. **Two-Point Crosses**
   - Determine if genes are linked
   - Calculate map distances
   - RF < 50% indicates linkage

3. **Three-Point Crosses**
   - Determine gene order efficiently
   - Map multiple genes simultaneously
   - Detect and quantify interference

4. **Interference**
   - One crossover affects probability of nearby crossovers
   - COC (Coefficient of Coincidence) = Observed DCO / Expected DCO
   - Interference = 1 - COC

**Applications:**
- Gene mapping in all organisms
- Understanding chromosome structure
- Breeding programs
- Human genetic counseling
- Evolutionary studies

---
## Additional Resources

**Further Reading:**
- Classical papers on genetic linkage by Thomas Hunt Morgan
- Modern genomic mapping techniques
- GWAS (Genome-Wide Association Studies)

**Try These Exercises:**
1. Use the simulators to create your own problems
2. Vary the parameters systematically to understand relationships
3. Compare two-point and three-point crosses for the same genes
4. Explore what happens with very tightly linked genes (RF < 5%)

---

**Created for Biology Students | Interactive Genetics Education**